# LangChain with Amazon Bedrock

LangChain is a framework that wraps model providers behind a common interface and adds building blocks like prompt templates and chains. This short module shows the same things we've done with the raw boto3 Converse API - invoke, prompt templates, conversation history - expressed the LangChain way, calling Bedrock underneath.

> **Teaching/Learning Tip:** frameworks like LangChain trade a bit of control for convenience and portability - swap `init_chat_model` targets and the same chain runs on a different provider. Whether that abstraction is worth it depends on your needs; this course standardizes on **Strands** for agents, so treat LangChain here as a concept survey, not the recommended path.

Requires `langchain`, `langchain-aws`, and `langchain-community` (in `requirements.txt`).

## Slide 1: Invoke a model

`init_chat_model` builds a chat model. We target Bedrock's Converse API. Messages are typed objects (`SystemMessage`, `HumanMessage`) rather than the raw dicts Converse uses.

> **Teaching/Learning Tip:** the slide uses `model="bedrock/anthropic.claude-sonnet-4-5-..."` with `model_provider="bedrock_converse"`. Claude models need an *inference profile*, so we use the `us.` prefix. The `"bedrock_converse:<id>"` string form is a compact way to say the same thing.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

model = init_chat_model(
    "bedrock_converse:us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name="us-east-1",
)

messages = [
    SystemMessage("Translate the following from English into Italian"),
    HumanMessage("hi!"),
]

response = model.invoke(messages)
print(response.content)

## Slide 2: What comes back (AIMessage)

LangChain returns an `AIMessage` object. The text is in `.content`; the Bedrock response metadata and token usage are attached too - the same information the raw Converse response carries, just wrapped in an object.

In [ ]:
print("content:      ", response.content)
print("usage:        ", response.usage_metadata)
print("stop reason:  ", response.response_metadata.get("stopReason"))
print("id:           ", response.id)

## Slide 3: Prompt templates

A `ChatPromptTemplate` has placeholders you fill in at call time - so the same template serves many inputs. Here `{language}` and `{text}` are filled per request.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"
prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

prompt = prompt_template.invoke({"language": "Italian", "text": "hi!"})
response = model.invoke(prompt)
print(response.content)

## Slides 4 & 5: A chain with conversation history in DynamoDB

We compose the prompt template and model into a **chain** (`prompt | model`), then wrap it so each turn's messages are loaded from and saved to DynamoDB, keyed by `session_id`. This is the LangChain equivalent of the persistent-history demo in `07_conversation_history`.

First, a session table (partition key `SessionId`, which `DynamoDBChatMessageHistory` expects).

In [ ]:
import boto3

REGION = "us-east-1"
TABLE = "LangchainSessionTable"
session = boto3.Session(region_name=REGION)
dynamodb = session.resource("dynamodb")

if TABLE not in [t.name for t in dynamodb.tables.all()]:
    print(f"Creating table '{TABLE}' (~20s) ...")
    t = dynamodb.create_table(
        TableName=TABLE,
        KeySchema=[{"AttributeName": "SessionId", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "SessionId", "AttributeType": "S"}],
        BillingMode="PAY_PER_REQUEST",
    )
    t.wait_until_exists()
print("Table ready")

> **Teaching/Learning Tip - region gotcha:** `DynamoDBChatMessageHistory` uses your default AWS region unless you pass a `boto3_session`. If your table is in `us-east-1` but your default region is elsewhere, you'll get a confusing `ResourceNotFoundException`. We pass an explicit session pinned to the region.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import DynamoDBChatMessageHistory

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])
chain = prompt_template | model

chain_with_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: DynamoDBChatMessageHistory(
        table_name=TABLE, session_id=session_id, boto3_session=session
    ),
    input_messages_key="question",
    history_messages_key="history",
)

In [ ]:
config = {"configurable": {"session_id": "alice-001"}}  # unique per user

r1 = chain_with_history.invoke({"question": "Hi! I'm Alice."}, config=config)
print("Turn 1:", r1.content)

r2 = chain_with_history.invoke({"question": "What's my name?"}, config=config)
print("Turn 2:", r2.content)

Turn 2 knows "Alice" because LangChain loaded the prior turn from DynamoDB and slotted it into the `history` placeholder - the same store-and-resend pattern as `07`, just handled by the framework.

> **Teaching/Learning Tip - deprecation reality:** running this raises two warnings - `langchain-community` is being sunset, and `RunnableWithMessageHistory` is deprecated in favor of LangGraph persistence. LangChain's abstractions move fast, which is a real maintenance cost. It's part of why this course standardizes on Strands for agent work.

## Cleanup

Delete the session table when done.

In [ ]:
dynamodb.Table(TABLE).delete()
print(f"Deleting table '{TABLE}' ...")